# Vérification des versions

In [ ]:
#version de python
import sys
print(sys.version)

In [ ]:
#version de pytorch
import torch
print(torch.__version__)

2.10.0+cpu


# Importation et préparation des données

Préparation usuelle des données

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving breast_test.xlsx to breast_test.xlsx
Saving breast_train.xlsx to breast_train.xlsx


In [ ]:


#importer les données
import pandas
pdTrain = pandas.read_excel("breast_train.xlsx",header=0)
print(pdTrain.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 399 entries, 0 to 398
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ucellsize   399 non-null    int64 
 1   ucellshape  399 non-null    int64 
 2   mgadhesion  399 non-null    int64 
 3   sepics      399 non-null    int64 
 4   normnucl    399 non-null    int64 
 5   mitoses     399 non-null    int64 
 6   classe      399 non-null    object
dtypes: int64(6), object(1)
memory usage: 21.9+ KB
None


In [ ]:
#premières lignes
pdTrain.head()

,ucellsize,ucellshape,mgadhesion,sepics,normnucl,mitoses,classe
0,1,1,1,2,1,1,begnin
1,3,2,1,3,6,1,begnin
2,9,7,3,4,7,1,malignant
3,10,10,7,10,2,1,malignant
4,8,8,4,10,1,1,malignant


In [ ]:
#structure avec les descripteurs
XTrain = pdTrain[pdTrain.columns[:-1]]
XTrain.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 399 entries, 0 to 398
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   ucellsize   399 non-null    int64
 1   ucellshape  399 non-null    int64
 2   mgadhesion  399 non-null    int64
 3   sepics      399 non-null    int64
 4   normnucl    399 non-null    int64
 5   mitoses     399 non-null    int64
dtypes: int64(6)
memory usage: 18.8 KB


In [ ]:
#outil pour centrer et réduire les X
#avec StandardScaler
from sklearn.preprocessing import StandardScaler
sts = StandardScaler()

#données centrées et réduites
ZTrain = sts.fit_transform(XTrain)
print(ZTrain.shape)

(399, 6)


In [ ]:
#distribution des classes
pdTrain.classe.value_counts()

,count
classe,
begnin,267
malignant,132


In [ ]:
#recoder la cible en 0 : begnin ; 1 : malignant
yTrain = pdTrain.classe.map({'malignant':1,'begnin':0}).values

#comptage
import numpy
numpy.unique(yTrain,return_counts=True)

(array([0, 1]), array([267, 132]))

Rendre les données compatibles avec PyTorch

In [ ]:
#tranformer les variables Z en tensor
tensor_ZTrain = torch.FloatTensor(ZTrain)

#qui est d'un type particulier
print(type(tensor_ZTrain))

In [ ]:
#dimensions
print(tensor_ZTrain.shape)

In [ ]:
#valeurs
print(tensor_ZTrain)

In [ ]:
#idem pour y
tensor_yTrain = torch.FloatTensor(yTrain)
print(tensor_yTrain)

In [ ]:
#dimension
print(tensor_yTrain.shape)

# Perceptron simple avec PyTorch

### Elaboration de la structure

Structure du réseau de neurones

In [ ]:
#classe de calcul - Perceptron simple
#héritier de torch.nn.Module
class MyPS(torch.nn.Module):
    #constructeur - liste des éléments
    #qui composent le réseau
    def __init__(self,p):
        #appel du constructeur de l'ancêtre
        super(MyPS,self).__init__()
        #couche d'entrée (p variables) vers sortie (1 neurone)
        self.layer1 = torch.nn.Linear(p,1)
        #fonction de transfert sigmoïde à la sortie
        self.ft1 = torch.nn.Sigmoid()

    #structuration des éléments
    #calcul de la sortie du réseau
    #à partir d'une matrice x en entrée
    def forward(self,x):
        #application de la combinaison linéaire
        comb_lin = self.layer1(x)
        #transformation avec la fonction de transfert
        proba = self.ft1(comb_lin)
        return proba

Choix de la fonction de perte à optimiser

In [ ]:
#fonction critère à optimiser
critere_ps = torch.nn.MSELoss()

Instanciation du modèle

In [ ]:
#instanciation du modele
#.shape[1] pour le nombre de descripteurs => 6
ps = MyPS(tensor_ZTrain.shape[1])

Choix de l'algorithme d'optimisation - Gradient stochastique

In [ ]:
#algorithme d'optimisation
#on lui passe les paramètres à manipuler
optimiseur_ps = torch.optim.Adam(ps.parameters())

Quelques vérifications - Coefficients et sortie du réseau

In [ ]:
#poids synaptiques
#initialisés aléatoirement
print(ps.layer1.weight)

In [ ]:
#et l'intercept
print(ps.layer1.bias)

In [ ]:
#calculer des sorties du réseau avec ces poids initiaux (aléatoires)
#équivalent à yPred = ps(tensor_ZTrain)
#on a des probas d'appartenance ici
yPred = ps.forward(tensor_ZTrain)

#affichage des 10 premières valeurs
print(yPred[:10])

In [ ]:
#format de yPred - matrice en réalité
print(yPred.shape)

In [ ]:
#pour transformer en vecteur
print(yPred.squeeze()[:10])

In [ ]:
#en effet
print(yPred.squeeze().shape)

Valeur de départ de la perte

In [ ]:
#MSE au départ (avec les poids initiaux aléatoires)
MSE1st = critere_ps(yPred.squeeze(),tensor_yTrain)
print(MSE1st)

In [ ]:
#vérification en passant par les vecteurs numpy
#on a bien une fonction de perte MSE
numpy.mean((yPred.squeeze().detach().numpy()-tensor_yTrain.numpy())**2)

### Entraînement du modèle

Bien détailler la structure du code avec les différentes étapes.

In [ ]:
#fonction pour apprentissage avec les paramètres :
#X, y, instance de classe torch, critère à optimiser, algo d'optimisation...
#...et n_epochs nombre de passage sur la base
def train_session(X,y,classifier,criterion,optimizer,n_epochs=10000):
    #vecteur pour collecter le loss au fil des itérations
    losses = numpy.zeros(n_epochs)
    #itérer (boucle) pour optimiser - n_epochs fois sur la base
    for iter in range(n_epochs):
        #réinitialiser (ràz) le gradient
        #nécessaire à chaque passage sinon PyTorch accumule
        optimizer.zero_grad()
        #calculer la sortie du réseau
        yPred = classifier.forward(X) #ou simplement classifier(X)
        #calculer la perte
        perte = criterion(yPred.squeeze(),y)
        #collecter la perte calculée dans le vecteur losses
        losses[iter] = perte.item()
        #calcul du gradient et retropropagation
        perte.backward()
        #màj des poids synaptiques
        optimizer.step()
    #sortie de la boucle
    #renvoie le vecteur avec les valeurs de perte à chaque epoch
    return losses

On peut lancer l'entraînement du modèle maintenant.

In [ ]:
#lancer l'apprentissage
#revenir sur les paramètres passés à la fonction
pertes = train_session(tensor_ZTrain,tensor_yTrain,ps,critere_ps,optimiseur_ps)

In [ ]:
#valeur de la perte au final
print(pertes[-1])

In [ ]:
#courbe de décroissance de la perte
import matplotlib.pyplot as plt
plt.plot(numpy.arange(0,pertes.shape[0]),pertes)
plt.title("Evolution fnct de perte")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.show()

In [ ]:
#inspection des coefficients après apprentissage
print(ps.layer1.weight)

In [ ]:
#et de l'intercept
print(ps.layer1.bias)

### Evaluation sur l'échantillon test

Préparation des données

In [ ]:
#chargement de l'échantillon test
pdTest = pandas.read_excel("breast_test.xlsx",header=0)
print(pdTest.info())

In [ ]:
#données centrées et réduites (avec les moyennes et écarts-type de train)
ZTest = sts.transform(pdTest[pdTest.columns[:-1]])
print(ZTest)

In [ ]:
#mettre au format tensor pour PyTorch
tensor_ZTest = torch.FloatTensor(ZTest)
print(tensor_ZTest)

In [ ]:
#recoder en 0/1 la cible
yTest = pdTest.classe.map({'malignant':1,'begnin':0}).values
yTest

In [ ]:
#prédiction de la probabilité d'appartenance
#en appliquant le modèle sur les descripteurs standardisés
proba_predTest = ps.forward(tensor_ZTest)
print(proba_predTest)

In [ ]:
#on a une matrice "torch"
proba_predTest.shape

In [ ]:
#que l'on transforme en vecteur numpy
vec_predTest = proba_predTest.detach().squeeze()
vec_predTest.shape

In [ ]:
#que l'on peut convertir en classe d'appartenance
#en comparant à la valeur 0.5
classe_predTest = numpy.where(vec_predTest > 0.5,1.0,0.0)

#on a un vecteur numpy ici
print(classe_predTest)

In [ ]:
#calcul de l'accuracy
print(numpy.mean(classe_predTest == yTest))

# Perceptron multicouche

Définir la nouvelle structure du réseau.

In [ ]:
#perceptron multicouche
class MyPMC(torch.nn.Module):
    #constructeur
    def __init__(self,p):
        #appel du constructeur de l'ancêtre
        super(MyPMC,self).__init__()
        #couche d'entrée (p variables) vers intermédiaire (2 neurones)
        self.layer1 = torch.nn.Linear(p,2)
        #fonction de transfert
        self.ft1 = torch.nn.Sigmoid()
        #couche intermédiaire vers sortie (1 neurone)
        self.layer2 = torch.nn.Linear(2,1)
        #fonction de transfert sigmoïde
        self.ft2 = torch.nn.Sigmoid()

    #premier forward
    #couche entrée -> couche cachée
    def forward_1(self,x):
        #application de la combinaison linéaire
        comb_lin_1 = self.layer1(x)
        #appliquer la fonction sigmoïde
        return self.ft1(comb_lin_1)

    #second forward
    #couche cachée -> couche sortie
    def forward_2(self,x_prim):
        #puis seconde combinaison linéaire
        comb_lin_2 = self.layer2(x_prim)
        #appliquer la transformation sigmoïde
        return self.ft2(comb_lin_2)


    #calcul de la sortie du réseau
    #à partir d'une matrice x en entrée
    def forward(self,x):
        #premier forward
        out_1 = self.forward_1(x)
        #second forward
        out_2 = self.forward_2(out_1)
        #return
        return out_2

Entraînement du modèle

In [ ]:
#instanciation
pmc = MyPMC(tensor_ZTrain.shape[1])

In [ ]:
#défintion des outils d'apprentissage
#critere - perte quadratique
critere_pmc = torch.nn.MSELoss()

#optimiseur
optimiseur_pmc = torch.optim.Adam(pmc.parameters())

In [ ]:
#lancement de l'entraînement
#en ré-explitant la fonction ci-dessus, seuls les paramètres changent
#dont l'instance de la structure du réseau "pmc"
pertes = train_session(tensor_ZTrain,tensor_yTrain,pmc,critere_pmc,optimiseur_pmc)

In [ ]:
#évolution de la perte vs. epoch
plt.plot(numpy.arange(0,pertes.shape[0]),pertes)
plt.title("Evolution fnct de perte")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.show()

Inspection des poids synaptiques (coefficients)

In [ ]:
#poids de entrée -> couche cachée
print(pmc.layer1.weight)

In [ ]:
#intercept corresp.
print(pmc.layer1.bias)

In [ ]:
#poids de couche cachée -> couche de sortie
print(pmc.layer2.weight)

In [ ]:
#intercept corresp.
print(pmc.layer2.bias)

Evaluation sur l'échantillon test

In [ ]:
#prédiction proba d'appartenance
proba_pmc = pmc.forward(tensor_ZTest)
print(proba_pmc)

In [ ]:
#conversion en classe d'appartenance
classe_pmc = numpy.where(proba_pmc.detach().squeeze() > 0.5,1.0,0.0)
print(classe_pmc)

In [ ]:
#accuracy
print(numpy.mean(classe_predTest == yTest))

Représentation intermédiaire de la couche cachée

In [ ]:
#obtenir la sortie de la couche cachée
#renvoyée par forward_1
hidden = pmc.forward_1(tensor_ZTrain)
print(hidden)

In [ ]:
#sous la forme d'un data frame pandas
p_hidden = pandas.DataFrame(hidden.detach().numpy(),columns=['F1','F2'])
p_hidden.head()

In [ ]:
#associer la classe d'apparenance au data frame
p_hidden['classe'] = pdTrain.classe
p_hidden.head()

In [ ]:
#affichage des points dans le plan "factoriel"
import seaborn as sns
sns.scatterplot(data=p_hidden,x='F1',y='F2',hue='classe')

In [ ]:
#rappel de la droite de séparation : couche cachée -> sortie
print(f"Coefficients : {pmc.layer2.weight}")
print(f"Intercept : {pmc.layer2.bias}")

In [ ]:
#récupération des coefficients et de l'intercept
coef = pmc.layer2.weight.detach().numpy()[0]
intercept = pmc.layer2.bias.detach().numpy()
print(coef)
print(intercept)

In [ ]:
#coordoonées de F2 quand F1 = 0
f2_0 = -intercept/coef[1]
print(f2_0)

In [ ]:
#coordoonées de F2 quand F1 = 1
f2_1 = (-intercept - coef[0] * 1)/coef[1]
print(f2_1)

In [ ]:
#droite de séparation dans l'espace intermédiaire
plt.plot(numpy.array([0,1]),numpy.array([f2_0,f2_1]),'k-')
sns.scatterplot(data=p_hidden,x='F1',y='F2',hue='classe')
plt.show()